# Similarity (Doc2Vec)

## Goal

So sánh cách xếp hạng câu bằng Doc2Vec (PV-DM, PV-DBOW) với cách làm bằng PhoBERT ở `phoBERT/04_similarity.ipynb`, dùng chung một query để hai kết quả có thể đối chiếu ranking (không đối chiếu giá trị cosine tuyệt đối, vì hai model nằm trên hai không gian vector khác nhau).

## Setup

Chạy notebook từ thư mục `backend/`, sau khi đã chạy `01_train_doc2vec.ipynb` và `02_document_embedding.ipynb`.

In [ ]:
from pathlib import Path

import torch
import torch.nn.functional as F
from gensim.models.doc2vec import Doc2Vec

## Load model + embeddings

In [ ]:
ARTIFACT_DIR = Path("artifacts")

dm_model = Doc2Vec.load(str(ARTIFACT_DIR / "doc2vec_dm.model"))
dbow_model = Doc2Vec.load(str(ARTIFACT_DIR / "doc2vec_dbow.model"))

dm_sentence_data = torch.load(ARTIFACT_DIR / "dm_sentence_embeddings.pt", weights_only=False)
dm_document_data = torch.load(ARTIFACT_DIR / "dm_document_embedding.pt", weights_only=False)
dbow_sentence_data = torch.load(ARTIFACT_DIR / "dbow_sentence_embeddings.pt", weights_only=False)
dbow_document_data = torch.load(ARTIFACT_DIR / "dbow_document_embedding.pt", weights_only=False)

print("PV-DM sentence embeddings:", dm_sentence_data["embeddings"].shape)
print("PV-DM document embedding: ", dm_document_data["embedding"].shape)
print("PV-DBOW sentence embeddings:", dbow_sentence_data["embeddings"].shape)
print("PV-DBOW document embedding: ", dbow_document_data["embedding"].shape)

## Corpus

Chỉ dùng để in lại câu gốc khi xếp hạng; thứ tự phải khớp với lúc train.

In [ ]:
# ruff: noqa: E501
segmented_sentences = [
    'Đặt vé từ TP HCM đi Singapore để công_tác , chị Hoàng_Loan , ở phường Xuân_Hoà , bất_ngờ vì mức giá lần đầu mua được kể từ sau đại_dịch " Năm_ngoái , chặng TP HCM - Singapore có lúc lên tới 3,2 triệu đồng một_chiều , còn năm nay tôi chỉ trả hơn 1,6 triệu đồng , đã gồm thuế , phí " , chị nói .',
    "Chiều về , vé cũng được áp_dụng mức giá khuyến_mại 19.000 đồng , nhưng sau khi cộng thuế , phí , tổng tiền chị Loan phải trả khoảng 2,4 triệu đồng .",
    "Theo chị , các khoản phí tại sân_bay Singapore cao hơn chiều bay từ Việt_Nam nên dù cùng giá vé niêm_yết , số tiền thực trả vẫn chênh_lệch đáng_kể .",
    "Tính cả hai chiều , chuyến đi Singapore của chị hết hơn 4 triệu đồng , giảm khoảng một_nửa so với cùng kỳ năm_ngoái .",
    "Sau Covid-19 , các đường_bay quốc_tế mất nhiều thời_gian để phục_hồi , trong khi nguồn cung chưa trở_lại như trước khiến giá luôn ở mức cao , nhất_là vào mùa du_lịch .",
    "Năm nay , nguồn cung tăng nhanh hơn , kéo_theo cạnh_tranh giữa các hãng và tạo thêm dư_địa giảm_giá .",
    "Mức giá chị Loan mua không phải trường_hợp cá_biệt .",
    "Khảo_sát các đường_bay từ TP HCM đi Singapore và Thái_Lan cho thấy mức giá khuyến_mại 19.000-90.000 đồng , chưa gồm thuế , phí chiếm đa_số các chặng bay trong tháng 8 và 9 .",
    "Sau khi cộng các khoản này , vé TP HCM - Singapore từ hơn 1,6 triệu đồng một_chiều , còn chặng TP HCM - Bangkok chưa đến 1,9 triệu đồng .",
    "Vé của một_số hãng hàng_không nước_ngoài trên cùng_đường bay hiện cao hơn khoảng 2-3 lần so với các hãng Việt_Nam .",
    "Từ Hà_Nội đi Singapore và Thái_Lan , giá vé của các hãng trong nước dao_động 2,6-3 triệu đồng một_chiều , đã gồm thuế , phí .",
    "Một_số ngày trong tháng 8 , mức thấp nhất còn hơn 2 triệu đồng .",
    "Với đường_bay TP HCM - Jakarta , giá cũng giảm nhưng mặt_bằng vẫn cao hơn Singapore và Thái_Lan .",
    "Nếu trước_đây vé khứ_hồi thường ở mức 7-10 triệu đồng , hiện giá thấp nhất khoảng 6,3 triệu đồng , đã gồm thuế , phí , tương_đương hơn 3 triệu đồng mỗi chiều .",
    "Mức giá cao hơn một phần do quãng đường_bay xa hơn .",
    "Các đường_bay từ Hà_Nội và TP HCM tới châu_Âu , Đông_Bắc_Á cũng giảm khoảng 10-15% so với trước .",
    "Giá đi xuống trong bối_cảnh nguồn cung hàng_không Việt_Nam tăng .",
    "Theo dữ_liệu dự_báo của Công_ty cung_cấp dữ_liệu hàng không OAG ( Anh ) , Việt_Nam có khoảng 7,3 triệu ghế cung_ứng trong tháng 8 , tăng 10% so với cùng kỳ năm_ngoái và đứng thứ hai Đông_Nam_Á , sau Indonesia .",
    "Trong khi tổng năng_lực khai_thác của thị_trường hàng_không Đông_Nam_Á tháng 8 chỉ tăng 0,8% so với cùng kỳ , nguồn cung của Việt_Nam tăng tới 10% .",
    "Trong đó , Vietnam_Airlines có khoảng 2,8 triệu ghế , tăng 8,2% , trong khi Vietjet khoảng 2,24 triệu ghế .",
    "Nguồn cung trên các đường_bay quốc_tế cũng được tăng_cường .",
    "Vietjet_Air cho biết , nâng tần_suất TP HCM - Kuala_Lumpur lên 7 chuyến mỗi tuần trong mùa cao_điểm , đồng_thời mở đường_bay TP HCM - Colombo từ ngày 18/8 .",
    "Hãng cũng chuẩn_bị khai_thác các đường_bay Hà_Nội - Almaty và Hà_Nội - Praha từ tháng 10 .",
    "Ông Hồng_Thanh , chủ một đại_lý vé máy_bay tại TP HCM , cho biết nguồn cung tăng và cạnh_tranh giữa các hãng là nguyên_nhân quan_trọng khiến giá vé quốc_tế hạ nhiệt .",
    "Các hãng phải tăng khuyến_mại , kích_cầu trong bối_cảnh sức_mua chưa phục_hồi như kỳ_vọng .",
    "Chi_phí nhiên_liệu cũng thuận_lợi hơn cho các hãng .",
    "Từ ngày 1/7 , Chính_phủ tiếp_tục kéo_dài thời_hạn áp_dụng thuế nhập_khẩu ưu_đãi , thuế bảo_vệ môi_trường và thuế_giá_trị gia_tăng với xăng_dầu , nhiên_liệu bay đến hết ngày 30/9/2026 , giúp giảm một phần chi_phí đầu_vào của các hãng hàng_không .",
    "Về nhu_cầu , thị_trường khách quốc_tế đến Việt_Nam tăng mạnh .",
    "Bảy tháng đầu năm , Việt_Nam đón gần 14 triệu lượt khách quốc_tế , tăng gần 14% so với cùng kỳ năm_ngoái .",
    "Riêng tháng 7 , lượng khách đạt khoảng 1,67 triệu lượt , trong đó đường_hàng không chiếm gần 83% .",
    "Nhu_cầu đi_lại quốc_tế tăng trong khi nguồn cung được bổ_sung khiến các hãng phải cạnh_tranh mạnh hơn để thu_hút khách .",
    "Đây cũng là một trong những yếu_tố kéo mặt_bằng giá xuống trong mùa hè năm nay .",
    "Không_chỉ quốc_tế , trước đó các hãng cũng liên_tục kích_cầu trên thị_trường nội_địa ngay giữa cao_điểm hè .",
    "Nhiều chương_trình đưa giá vé một_số chặng về mức 0 đồng hoặc vài chục nghìn đồng , chưa gồm thuế , phí .",
]

print(f"Number of sentences: {len(segmented_sentences)}")

## Query

Dùng chung query với `phoBERT/04_similarity.ipynb` để so sánh hành vi ranking giữa hai phương pháp.

In [ ]:
def tokenize(sentence: str) -> list[str]:
    """Tách theo khoảng trắng; corpus đã segment nên giữ nguyên từ ghép nối bằng '_' làm một token."""
    return sentence.split()


query = "Giá vé máy_bay đi Singapore giảm mạnh ."
query_tokens = tokenize(query)

print(f"Query: {query}")
print(f"Tokens: {query_tokens}")

## Infer query vector

`infer_vector` chạy gradient descent trên token cố định để suy ra một paragraph vector mới, nằm trong cùng không gian đã train của mỗi model — tương tự việc PhoBERT encode một câu query chưa từng thấy.

In [ ]:
def infer_query_vector(model: Doc2Vec, tokens: list[str], epochs: int = 200) -> torch.Tensor:
    """infer_vector suy ra paragraph vector mới cho token cố định, cùng không gian đã train."""
    vector = model.infer_vector(tokens, epochs=epochs)
    return torch.tensor(vector, dtype=torch.float32)


dm_query_embedding = infer_query_vector(dm_model, query_tokens)
dbow_query_embedding = infer_query_vector(dbow_model, query_tokens)

# V1 - Sentence-level (MAX)

Document Score = `max(sentence similarity)`, giống baseline V1 ở `phoBERT/04_similarity.ipynb`.

In [ ]:
def rank_sentences(
    sentence_embeddings: torch.Tensor, query_embedding: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    """Cosine similarity giữa query và từng câu, kèm thứ tự rank giảm dần."""
    scores = F.cosine_similarity(sentence_embeddings, query_embedding.unsqueeze(0), dim=1)
    ranking = torch.argsort(scores, descending=True)
    return scores, ranking

### PV-DM

In [ ]:
dm_scores, dm_ranking = rank_sentences(dm_sentence_data["embeddings"], dm_query_embedding)

for rank, sentence_index in enumerate(dm_ranking.tolist(), start=1):
    score = dm_scores[sentence_index].item()
    print(f"{rank}. score={score:.4f} | sentence_id={sentence_index}")
    print(f"   {segmented_sentences[sentence_index]}\n")

dm_v1_document_score = dm_scores.max().item()
print(f"PV-DM V1 document score: {dm_v1_document_score:.4f}")

### PV-DBOW

In [ ]:
dbow_scores, dbow_ranking = rank_sentences(dbow_sentence_data["embeddings"], dbow_query_embedding)

for rank, sentence_index in enumerate(dbow_ranking.tolist(), start=1):
    score = dbow_scores[sentence_index].item()
    print(f"{rank}. score={score:.4f} | sentence_id={sentence_index}")
    print(f"   {segmented_sentences[sentence_index]}\n")

dbow_v1_document_score = dbow_scores.max().item()
print(f"PV-DBOW V1 document score: {dbow_v1_document_score:.4f}")

# V2 - Document-level vector

Cosine similarity giữa query và vector duy nhất của cả bài (`infer_vector` trên toàn bộ token), giống baseline V2 ở `phoBERT/04_similarity.ipynb`.

In [ ]:
dm_v2_document_score = F.cosine_similarity(
    dm_query_embedding.unsqueeze(0), dm_document_data["embedding"].unsqueeze(0), dim=1
).item()
dbow_v2_document_score = F.cosine_similarity(
    dbow_query_embedding.unsqueeze(0), dbow_document_data["embedding"].unsqueeze(0), dim=1
).item()

print(f"PV-DM V2 document score: {dm_v2_document_score:.4f}")
print(f"PV-DBOW V2 document score: {dbow_v2_document_score:.4f}")

# Compare

In [ ]:
print("=== RESULT ===")
print(f"Query:   {query}")
print()

print(f"PV-DM   - Sentence-level (MAX): {dm_v1_document_score:.4f}")
print(f"PV-DM   - Document-level:       {dm_v2_document_score:.4f}")
print(f"PV-DBOW - Sentence-level (MAX): {dbow_v1_document_score:.4f}")
print(f"PV-DBOW - Document-level:       {dbow_v2_document_score:.4f}")

## Notes

- Giá trị cosine ở đây **không so sánh trực tiếp được** với `phoBERT/04_similarity.ipynb`: PhoBERT là model pretrained trên corpus khổng lồ, còn Doc2Vec ở đây train từ đầu (from scratch) trên 34 câu — chỉ nên so sánh **thứ tự ranking** (câu nào được xếp hạng cao nhất), không so sánh magnitude.
- Muốn Doc2Vec cạnh tranh được với PhoBERT về chất lượng, cần train trên toàn bộ corpus thật (`make crawl` + `make preprocess` + `make segment`) với mỗi bài báo là một document, thay vì demo 34 pseudo-document như notebook này.